In [1]:
import eval_utils
from PIL import Image
import torchvision.transforms.functional as TF

/home/ahc/miniconda3/envs/relax-flow/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
image = Image.open("./data/A_bike_with_a_blue_front_wheel_and_a_red_rear_wheel/image.png")
mask = Image.open("./data/A_bike_with_a_blue_front_wheel_and_a_red_rear_wheel/mask.png")
prior1 = Image.open("./data/A_bike_with_a_blue_front_wheel_and_a_red_rear_wheel/prior.png")
masked_image = Image.composite(image, Image.new('RGBA', image.size, (0,0,0,0)), mask)

# prior1 = Image.open("./eval_data/duck/A yellow rubber duck with jetpack, white background, 3D render.png")
masked_image = Image.open("./eval_data/duck/gt_obs_segmented/frame_000000.png")

masked_image = Image.open("./eval_data/chair/gt_obs_segmented/frame_000000.png")


a = eval_utils.clip_text_similarity([image], "Vehicle")
b = eval_utils.clip_text_similarity([masked_image], "Vehicle")

c = eval_utils.clip_text_similarity([image], "Chair")
d = eval_utils.clip_text_similarity([masked_image], "Chair")

print(a,b)
print(c,d)

0.21926526725292206 0.14019905030727386
0.16766014695167542 0.24529904127120972


In [3]:

from pathlib import Path
from PIL import Image

EVAL_DATA = Path("./eval_data")
MODELS = ["SAM3D", "TRELLIS", "TRELLIS.2"]

def load_images_from_dir(folder: Path) -> list[Image.Image]:
    imgs = []
    for f in sorted(folder.iterdir()):
        if f.suffix.lower() in (".png", ".jpg", ".jpeg"):
            imgs.append(Image.open(f).convert("RGBA"))
    return imgs

def get_pred_images(pred_dir: Path, model: str) -> list[Image.Image] | None:
    model_dir = pred_dir / model
    if not model_dir.exists():
        return None
    if model == "SAM3D":
        relaxflow = model_dir / "relaxflow" / "frames_rgba"
        if not relaxflow.exists():
            return None
        imgs = load_images_from_dir(relaxflow)
    else:
        imgs = load_images_from_dir(model_dir)
    return imgs if imgs else None

def get_pred_text(pred_dir: Path) -> str | None:
    """Filename (sans extension) of the single image directly inside predX/."""
    for f in pred_dir.iterdir():
        if f.is_file() and f.suffix.lower() in (".png", ".jpg", ".jpeg"):
            return f.stem
    return None

scores = {m: {"clip_text": [], "clip_img": []} for m in MODELS}

for scene_dir in sorted(EVAL_DATA.iterdir()):
    if not scene_dir.is_dir():
        continue
    gt_obs_seg_dir = scene_dir / "gt_obs_segmented"
    if not gt_obs_seg_dir.exists():
        continue
    gt_images = load_images_from_dir(gt_obs_seg_dir)
    if not gt_images:
        continue
    observed_render = gt_images[0]

    pred_dirs = sorted([d for d in scene_dir.iterdir() if d.is_dir() and d.name.startswith("pred")])

    for pred_dir in pred_dirs:
        scene_text = get_pred_text(pred_dir)
        if scene_text is None:
            print(f"  skip (no prompt file)  {scene_dir.name}/{pred_dir.name}")
            continue

        print(">>>>>", scene_text)

        for model in MODELS:
            pred_imgs = get_pred_images(pred_dir, model)
            if pred_imgs is None:
                print(f"  skip  {scene_dir.name}/{pred_dir.name}/{model}")
                continue

            ct = eval_utils.clip_text_similarity(pred_imgs, scene_text)
            ci = eval_utils.clip_img_similarity(gt_images, observed_render, pred_imgs)
            scores[model]["clip_text"].append(ct)
            scores[model]["clip_img"].append(ci)
            print(f"  {scene_dir.name}/{pred_dir.name}/{model}  text={ct:.4f}  img={ci:.4f}")

print("\n=== Average scores by model ===")
for model in MODELS:
    ct_vals = scores[model]["clip_text"]
    ci_vals = scores[model]["clip_img"]
    if ct_vals:
        print(f"{model:12s}  clip_text={sum(ct_vals)/len(ct_vals):.4f} (n={len(ct_vals)})  "
              f"clip_img={sum(ci_vals)/len(ci_vals):.4f} (n={len(ci_vals)})")
    else:
        print(f"{model:12s}  no data")


>>>>> A yellow rubber duck with a pirate eye patch on its left, white background, centered, 3D render
  DUCK/pred0/SAM3D  text=0.3383  img=0.8514
  DUCK/pred0/TRELLIS  text=0.3227  img=0.8870
  DUCK/pred0/TRELLIS.2  text=0.3190  img=0.8604
>>>>> A yellow rubber duck with left eye blue right eye white, white background, centered, 3D render
  skip  DUCK/pred1/SAM3D
  DUCK/pred1/TRELLIS  text=0.3167  img=0.8975
  DUCK/pred1/TRELLIS.2  text=0.3062  img=0.8941
>>>>> A yellow rubber duck with the left eye replaced by a googlie eye, white background, centered, 3D render
  DUCK/pred2/SAM3D  text=0.2928  img=0.8614
  DUCK/pred2/TRELLIS  text=0.2882  img=0.9004
  DUCK/pred2/TRELLIS.2  text=0.2933  img=0.8934
>>>>> Red flip paper calendar with many binder rings showing month of March, white background, centered, 3D render
  skip  FLIP_CALENDAR_RED/pred0/SAM3D
  FLIP_CALENDAR_RED/pred0/TRELLIS  text=0.2514  img=0.8319
  skip  FLIP_CALENDAR_RED/pred0/TRELLIS.2
>>>>> Red flip paper calendar with man